In [1]:
import numpy as np

In [2]:
def load_dense_demag(filename, ntot, dtype=np.float32):
    """
    Load the 6 dense demag tensor matrices written by Fortran direct-access file.

    Returns:
        Kxx, Kxy, Kxz, Kyy, Kyz, Kzz  (each shape [ntot, ntot])
    """
    n_elem = ntot * ntot
    rec_bytes = n_elem * np.dtype(dtype).itemsize

    mats = []

    with open(filename, "rb") as f:
        for rec in range(6):
            f.seek(rec * rec_bytes)
            data = np.fromfile(f, dtype=dtype, count=n_elem)
            mat = data.reshape((ntot, ntot), order="F")  # Fortran layout
            mats.append(mat)

    return mats


In [3]:
cu_ref_Kxx, cu_ref_Kxy, cu_ref_Kxz, cu_ref_Kyy, cu_ref_Kyz, cu_ref_Kzz = load_dense_demag(
    "CUDA_dense_K_matrices.bin", ntot=36*9, dtype=np.float32
)

ref_Kxx, ref_Kxy, ref_Kxz, ref_Kyy, ref_Kyz, ref_Kzz = load_dense_demag(
    "dense_std_ref.bin", ntot=36*9, dtype=np.float32
)

omp1_Kxx, omp1_Kxy, omp1_Kxz, omp1_Kyy, omp1_Kyz, omp1_Kzz = load_dense_demag(
    "dense_std_omp1_2.bin", ntot=36*9, dtype=np.float32
)

omp2_Kxx, omp2_Kxy, omp2_Kxz, omp2_Kyy, omp2_Kyz, omp2_Kzz = load_dense_demag(
    "dense_std_omp2_2.bin", ntot=36*9, dtype=np.float32
)

In [4]:
arrays = ['Kxx', 'Kxy', 'Kxz', 'Kyy', 'Kyz', 'Kzz']
for array in arrays:
    ref_array = globals()[f"ref_{array}"]
    omp1_array = globals()[f"cu_ref_{array}"]
    identical = np.array_equal(ref_array, omp1_array)
    print(f"{array}: {'Identical' if identical else 'Different'}")

Kxx: Identical
Kxy: Identical
Kxz: Identical
Kyy: Identical
Kyz: Identical
Kzz: Identical
